# Project 1 ANSC 4040 Model Tuning

This is a notebook for Project 1 that will be used for expanding the same size and tuning the hyperparameters of original model

In [5]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Train and Validation Sets (100 cows)

In [8]:
train_path = "train_data.csv"
val_path = "val_data.csv"

In [10]:
# Find the training-set cow IDs using a bounded-memory scan.
animal_ids = set()
for id_chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000,
):
    animal_ids.update(id_chunk["AnimalId"].dropna().unique())

selected_animals = set(
    pd.Series(sorted(animal_ids)).sample(n=100, random_state=42).tolist()
)

# Read only model columns and rows for those 100 cows.
model_cols = [
    "AnimalId",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

def load_selected_cows(file_path, selected_ids):
    matching_chunks = []
    for chunk in pd.read_csv(
        file_path,
        usecols=model_cols,
        dtype={"AnimalId": "string"},
        chunksize=100_000,
    ):
        matching = chunk[chunk["AnimalId"].isin(selected_ids)]
        if not matching.empty:
            matching_chunks.append(matching)
    if not matching_chunks:
        raise ValueError(f"None of the selected cows were found in {file_path}.")
    return pd.concat(matching_chunks, ignore_index=True)

train = load_selected_cows(train_path, selected_animals)
val = load_selected_cows(val_path, selected_animals)

print(f"Selected cows: {sorted(selected_animals)}")
print(f"Training rows: {len(train):,}; validation rows: {len(val):,}")

# Split EventDate into numerical calendar features
for data in (train, val):
    event_date = pd.to_datetime(data["EventDate"])
    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

Selected cows: ['-1.1716852323791606e+18', '-1.1833355166981594e+18', '-1.3332775441306954e+17', '-2.05144505093895e+18', '-2.099013255094703e+18', '-2.221464130034193e+18', '-2.25327613911556e+18', '-2.2990285093312376e+18', '-2.3422767415993856e+18', '-2.362711626198422e+18', '-2.4428249858373806e+18', '-2.49718279280027e+18', '-2.5715480822330004e+18', '-2.972840480631011e+17', '-3.1150506377499264e+18', '-3.2709915003359217e+18', '-3.455482512856752e+18', '-3.671253680800321e+18', '-3.7808412218609434e+18', '-3.788877104804816e+18', '-3.85672259016968e+18', '-4.1398554415718595e+18', '-4.225393355328677e+18', '-4.278792536301574e+18', '-4.4539072266346624e+18', '-4.482367362136449e+18', '-4.572979853174115e+18', '-4.725483000184784e+18', '-4.7479871501442355e+17', '-4.907405775221401e+18', '-5.274629919485563e+18', '-5.297614520013029e+17', '-5.398349683912395e+18', '-5.422953988785325e+18', '-5.451041495972229e+18', '-5.501116040247302e+18', '-5.556540390544463e+18', '-6.073988599

# Tuned Hyperparameters

In [ ]:
# Target variable
target = "AnimalId"

# Define feature by including columns
feature_cols = ["Avgmilkflow", "EventYear", "EventMonth", "EventDay", "EventDayOfWeek", "Flow30_60Session", "YieldFirst2Min_Session", "YieldSession", "DurationSession_sec", "milking"]



X_train = train[feature_cols]
y_train = train[target]

X_val = val[feature_cols]
y_val = val[target]

rf_model = RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        max_depth=5,
        min_samples_leaf=1,
        min_samples_split=2
    )

# Fit model
rf_model.fit(X_train, y_train)

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [14]:
# Evaluate on validation rows that were not used to fit the model.
y_val_pred = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation accuracy: {val_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation accuracy: 8.49%

Classification report:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.00      0.00      0.00       228
-1.1833355166981594e+18       0.03      0.11      0.05        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.06      0.28      0.10       121
 -2.099013255094703e+18       0.05      0.08      0.06       315
 -2.221464130034193e+18       0.00      0.00      0.00       244
  -2.25327613911556e+18       0.05      0.24      0.08       147
-2.2990285093312376e+18       0.40      0.15      0.22       243
-2.3422767415993856e+18       0.00      0.00      0.00        45
 -2.362711626198422e+18       0.00      0.00      0.00       233
-2.4428249858373806e+18       0.00      0.00      0.00        99
  -2.49718279280027e+18       0.00      0.00      0.00       168
-2.5715480822330004e+18       0.00      0.00      0.00       209
 -2.972840480631011e+17       0.00    

# Hyperparameter tuning (now for 100 cows)

This small grid searches a realistic set of Random Forest settings to find the best combination without making the tuning process too slow or too broad.

In [15]:
from sklearn.model_selection import GridSearchCV, KFold

# Hyperparameter tuning with GridSearchCV
# This explores a small grid of RandomForest settings and selects the combination
# with the best cross-validation accuracy.

param_grid = {
    "n_estimators": [20, 40],
    "max_depth": [5, 10],
    "min_samples_leaf": [4, 8],
}

cv = KFold(n_splits=2, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=1),
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")
print(f"Best parameters: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_val_pred_best = best_model.predict(X_val)
best_val_accuracy = accuracy_score(y_val, y_val_pred_best)

print(f"Validation accuracy with tuned model: {best_val_accuracy:.2%}")
print("\nClassification report for tuned model:")
print(classification_report(y_val, y_val_pred_best, zero_division=0))

results_df = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)
print("\nTop GridSearchCV results:")
print(results_df[["params", "mean_test_score", "std_test_score"]].head(10).to_string(index=False))

Fitting 2 folds for each of 8 candidates, totalling 16 fits
Best cross-validation accuracy: 46.70%
Best parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 40}
Validation accuracy with tuned model: 10.82%

Classification report for tuned model:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.30      0.42      0.35       228
-1.1833355166981594e+18       0.13      0.86      0.22        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.03      0.17      0.05       121
 -2.099013255094703e+18       0.00      0.00      0.00       315
 -2.221464130034193e+18       0.00      0.00      0.00       244
  -2.25327613911556e+18       0.06      0.25      0.10       147
-2.2990285093312376e+18       0.12      0.02      0.03       243
-2.3422767415993856e+18       1.00      0.31      0.47        45
 -2.362711626198422e+18       0.13      0.01      0.02       233
-2.4428249858373806e+1

# 100 cow tuned hyperparameters for RandomForestClassifier

100- {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 40}

# Explore XGBoost as an alternative tree-based model

In [1]:
%pip install xgboost

  Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
  Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl (2.4 MB)
Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl (5.4 MB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [xgboost]m2/3 [xgboost]
Note: you may need to restart the kernel to use updated packages.


In [18]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier

# XGBoost requires consecutive integer labels; keep the Random Forest labels intact.
animal_ids_sorted = sorted(selected_animals)
animal_id_to_idx = {animal_id: i for i, animal_id in enumerate(animal_ids_sorted)}
y_train_xgb = train["AnimalId"].map(animal_id_to_idx)
y_val_xgb = val["AnimalId"].map(animal_id_to_idx)
if y_train_xgb.isna().any() or y_val_xgb.isna().any():
    raise ValueError("Found an AnimalId that is missing from the XGBoost label mapping.")
y_train_xgb = y_train_xgb.astype("int64")
y_val_xgb = y_val_xgb.astype("int64")

# Use the Random Forest fit's training/validation data and a modest tree count.
xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    n_estimators=40,
    max_depth=4,
    min_child_weight=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,
)
xgb_model.fit(X_train, y_train_xgb)

y_val_pred_xgb = xgb_model.predict(X_val)
xgb_val_accuracy = accuracy_score(y_val_xgb, y_val_pred_xgb)
xgb_val_macro_f1 = f1_score(y_val_xgb, y_val_pred_xgb, average="macro", zero_division=0)
print(f"XGBoost validation accuracy: {xgb_val_accuracy:.2%}")
print(f"XGBoost validation macro-F1: {xgb_val_macro_f1:.2%}")
print(classification_report(
    y_val_xgb,
    y_val_pred_xgb,
    labels=range(len(animal_ids_sorted)),
    target_names=[str(animal_id) for animal_id in animal_ids_sorted],
    zero_division=0,
))

XGBoost validation accuracy: 8.85%
XGBoost validation macro-F1: 9.18%
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.30      0.27      0.29       228
-1.1833355166981594e+18       0.10      0.89      0.18        28
-1.3332775441306954e+17       1.00      0.02      0.03        59
  -2.05144505093895e+18       0.02      0.07      0.03       121
 -2.099013255094703e+18       0.00      0.00      0.00       315
 -2.221464130034193e+18       0.02      0.00      0.01       244
  -2.25327613911556e+18       0.04      0.16      0.06       147
-2.2990285093312376e+18       0.04      0.00      0.01       243
-2.3422767415993856e+18       1.00      0.31      0.47        45
 -2.362711626198422e+18       0.00      0.00      0.00       233
-2.4428249858373806e+18       0.00      0.00      0.00        99
  -2.49718279280027e+18       0.00      0.00      0.00       168
-2.5715480822330004e+18       0.00      0.00      0.00       209
 -2.972840480631011

In [20]:
import numpy as np
from sklearn.model_selection import GridSearchCV

# Make two forward-in-time folds within each animal, matching the project's
# chronological train/validation split and avoiding random-row leakage.
cv_rows = train[["AnimalId", "EventDate"]].copy()
cv_rows["_position"] = np.arange(len(train))
cv_rows["_date"] = pd.to_datetime(cv_rows["EventDate"])
cv_rows = cv_rows.sort_values(["AnimalId", "_date"])
cv_rows["_order"] = cv_rows.groupby("AnimalId").cumcount()
cv_rows["_count"] = cv_rows.groupby("AnimalId")["_position"].transform("size")
cv_rows["_fraction"] = (cv_rows["_order"] + 1) / cv_rows["_count"]

cv_splits = []
for train_cutoff, validation_cutoff in ((0.60, 0.80), (0.80, 1.01)):
    is_training = (cv_rows["_fraction"] <= train_cutoff) | (cv_rows["_order"] == 0)
    is_validation = (
        (cv_rows["_fraction"] > train_cutoff)
        & (cv_rows["_fraction"] <= validation_cutoff)
        & (cv_rows["_order"] > 0)
    )
    train_indices = cv_rows.loc[is_training, "_position"].to_numpy(dtype=int)
    validation_indices = cv_rows.loc[is_validation, "_position"].to_numpy(dtype=int)
    if len(train_indices) == 0 or len(validation_indices) == 0:
        raise ValueError("A chronological XGBoost CV fold is empty.")
    cv_splits.append((train_indices, validation_indices))

# Compare eight meaningfully different settings (16 fits total), sequentially.
xgb_param_grid = {
    "n_estimators": [40, 80],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}
xgb_search = GridSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        min_child_weight=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=1,
    ),
    param_grid=xgb_param_grid,
    scoring="accuracy",
    cv=cv_splits,
    n_jobs=1,
    verbose=1,
)
xgb_search.fit(X_train, y_train_xgb)

# GridSearchCV refits the best setting on all training data by default.
tuned_xgb_model = xgb_search.best_estimator_
y_val_pred_tuned = tuned_xgb_model.predict(X_val)
tuned_xgb_val_accuracy = accuracy_score(y_val_xgb, y_val_pred_tuned)
tuned_xgb_val_macro_f1 = f1_score(
    y_val_xgb, y_val_pred_tuned, average="macro", zero_division=0
)

print(f"Best XGBoost CV accuracy: {xgb_search.best_score_:.2%}")
print(f"Best XGBoost hyperparameters: {xgb_search.best_params_}")
print(f"Baseline XGBoost validation accuracy: {xgb_val_accuracy:.2%}")
print(f"Tuned XGBoost validation accuracy: {tuned_xgb_val_accuracy:.2%}")
print(f"Tuned XGBoost validation macro-F1: {tuned_xgb_val_macro_f1:.2%}")

accuracy_change = tuned_xgb_val_accuracy - xgb_val_accuracy
print(f"Validation accuracy change: {accuracy_change:+.2%} (percentage points)")
if accuracy_change > 0:
    print("Result: hyperparameter tuning improved validation accuracy.")
elif accuracy_change < 0:
    print("Result: hyperparameter tuning reduced validation accuracy.")
else:
    print("Result: hyperparameter tuning did not change validation accuracy.")

Fitting 2 folds for each of 8 candidates, totalling 16 fits
Best XGBoost CV accuracy: 10.63%
Best XGBoost hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 80}
Baseline XGBoost validation accuracy: 8.85%
Tuned XGBoost validation accuracy: 9.78%
Tuned XGBoost validation macro-F1: 9.55%


# Conclusion: the tuned accuracy of XGBC is worse than RandomForestGenerator

# Return to RandomForest for n=400 cows

This fits a fresh model on 400 cow IDs shared by the training and validation files, then reports accuracy on validation rows not used for fitting.

In [11]:
# Select 400 cow IDs present in both splits so every validation class is known during training.
validation_animal_ids = set()
for id_chunk in pd.read_csv(
    val_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000,
):
    validation_animal_ids.update(id_chunk["AnimalId"].dropna().unique())

shared_animal_ids = sorted(animal_ids & validation_animal_ids)
if len(shared_animal_ids) < 400:
    raise ValueError(
        f"Need at least 400 cows present in both splits; found {len(shared_animal_ids)}."
    )

selected_animals_400 = set(
    pd.Series(shared_animal_ids, dtype="string").sample(n=400, random_state=42).tolist()
)
train_400 = load_selected_cows(train_path, selected_animals_400)
val_400 = load_selected_cows(val_path, selected_animals_400)

for data in (train_400, val_400):
    event_date = pd.to_datetime(data["EventDate"])
    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

if train_400["AnimalId"].nunique() != 400 or val_400["AnimalId"].nunique() != 400:
    raise ValueError("The 400-cow train and validation subsets must each contain all 400 cows.")

X_train_400 = train_400[feature_cols]
y_train_400 = train_400["AnimalId"]
X_val_400 = val_400[feature_cols]
y_val_400 = val_400["AnimalId"]

rf_400 = RandomForestClassifier(
    n_estimators=40,
    max_depth=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=1,
)
rf_400.fit(X_train_400, y_train_400)

y_val_pred_400 = rf_400.predict(X_val_400)
rf_400_val_accuracy = accuracy_score(y_val_400, y_val_pred_400)

print(f"Cow IDs in both train and validation: {len(selected_animals_400)}")
print(f"Training rows: {len(train_400):,}; validation rows: {len(val_400):,}")
print(f"Random Forest validation accuracy (400 cows): {rf_400_val_accuracy:.2%}")

Cow IDs in both train and validation: 400
Training rows: 199,024; validation rows: 56,859
Random Forest validation accuracy (400 cows): 4.29%


In [13]:
# Tune a small, serial set of Random Forest settings using a chronological holdout
# from training data. Keep the final validation set untouched until model selection.
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

tuning_rows = train_400[["AnimalId", "EventDate"]].copy()
tuning_rows["_position"] = np.arange(len(tuning_rows))
tuning_rows["_date"] = pd.to_datetime(tuning_rows["EventDate"])
tuning_rows = tuning_rows.sort_values(["AnimalId", "_date"])
tuning_rows["_row_in_cow"] = tuning_rows.groupby("AnimalId").cumcount()
tuning_rows["_cow_rows"] = tuning_rows.groupby("AnimalId")["_position"].transform("size")
tuning_rows["_cutoff"] = (tuning_rows["_cow_rows"] * 0.8).astype(int).clip(lower=1)
tune_validation_positions = tuning_rows.loc[
    tuning_rows["_row_in_cow"] >= tuning_rows["_cutoff"], "_position"
].to_numpy(dtype=int)
tune_training_positions = tuning_rows.loc[
    tuning_rows["_row_in_cow"] < tuning_rows["_cutoff"], "_position"
].to_numpy(dtype=int)

X_tune_train = X_train_400.iloc[tune_training_positions]
y_tune_train = y_train_400.iloc[tune_training_positions]
X_tune_val = X_train_400.iloc[tune_validation_positions]
y_tune_val = y_train_400.iloc[tune_validation_positions]
if y_tune_train.nunique() != 400 or y_tune_val.empty:
    raise ValueError("The chronological tuning split must train on all 400 cows and have held-out rows.")

parameter_distributions = {
    "n_estimators": [40, 60],
    "max_depth": [10, 16, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [2, 4, 8],
    "max_features": ["sqrt", 0.5],
}

# One chronological CV split; n_jobs=1 and pre_dispatch=1 limit resource use.
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=1),
    param_distributions=parameter_distributions,
    n_iter=5,
    scoring="accuracy",
    cv=[(tune_training_positions, tune_validation_positions)],
    refit=True,
    random_state=42,
    n_jobs=1,
    pre_dispatch=1,
    verbose=1,
)
random_search.fit(X_train_400, y_train_400)

best_rf_400 = random_search.best_estimator_
best_rf_params_400 = random_search.best_params_
best_y_val_pred_400 = best_rf_400.predict(X_val_400)
best_rf_400_val_accuracy = accuracy_score(y_val_400, best_y_val_pred_400)

print(f"Best chronological tuning accuracy: {random_search.best_score_:.2%}")
print(f"Best parameters: {best_rf_params_400}")
print(f"Randomized-search validation accuracy (400 cows): {best_rf_400_val_accuracy:.2%}")

Fitting 1 folds for each of 5 candidates, totalling 5 fits
Best chronological tuning accuracy: 5.64%
Best parameters: {'n_estimators': 60, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 20}
Randomized-search validation accuracy (400 cows): 4.94%
